Below is **production-style hallucination detection code** used in **RAG systems**.
This is **interview-ready**, **practical**, and **easy to explain**.

I’ll show **3 levels** (simple → strong → enterprise).

---

# 🔍 Hallucination Detection in RAG (with Code)

## Scenario

We already have:

* **User query**
* **Retrieved documents**
* **LLM answer**

Goal 👉 **Verify the answer is grounded in retrieved context**

---

# ✅ Level 1: Keyword / Sentence Grounding Check (Simple & Fast)

### 🔹 Idea

If key sentences in the answer **don’t exist in retrieved context**, it’s likely hallucinated.

### ✅ Code

```python
def keyword_grounding_check(answer, docs):
    context = " ".join(d.page_content.lower() for d in docs)
    answer_sentences = answer.lower().split(".")

    supported = 0
    for sentence in answer_sentences:
        if sentence.strip() and sentence.strip() in context:
            supported += 1

    return supported / max(len(answer_sentences), 1)
```

### 🔹 Usage

```python
score = keyword_grounding_check(answer, retrieved_docs)

if score < 0.5:
    print("❌ Possible hallucination detected")
else:
    print("✅ Answer is grounded")
```

### 🎯 Interview Line

> “This is a fast heuristic used before expensive validation.”

---

# ✅ Level 2: Semantic Similarity Validation (Recommended)

### 🔹 Idea

Check **semantic similarity** between:

* Answer
* Retrieved context

### ✅ Code

```python
from langchain_openai import OpenAIEmbeddings
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

embeddings = OpenAIEmbeddings()

def semantic_faithfulness_check(answer, docs):
    context = " ".join(d.page_content for d in docs)

    answer_vec = embeddings.embed_query(answer)
    context_vec = embeddings.embed_query(context)

    similarity = cosine_similarity(
        [answer_vec], [context_vec]
    )[0][0]

    return similarity
```

### 🔹 Usage

```python
score = semantic_faithfulness_check(answer, retrieved_docs)

if score < 0.75:
    print("❌ Hallucination detected (low similarity)")
else:
    print("✅ Answer supported by context")
```

### 🎯 Interview Line

> “We use semantic similarity to detect hallucinations beyond exact text matching.”

---

# ✅ Level 3: LLM-Based Faithfulness Judge (Enterprise-Grade)

### 🔹 Idea

Use an LLM to **judge whether the answer is supported by context**

### ✅ Code

```python
from langchain_openai import ChatOpenAI

judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def llm_faithfulness_check(answer, docs):
    context = "\n".join(d.page_content for d in docs)

    prompt = f"""
    Determine whether the answer is fully supported
    by the provided context.

    Context:
    {context}

    Answer:
    {answer}

    Reply only YES or NO.
    """

    verdict = judge_llm.invoke(prompt).content.strip()
    return verdict == "YES"
```

### 🔹 Usage

```python
if llm_faithfulness_check(answer, retrieved_docs):
    print("✅ Answer is faithful")
else:
    print("❌ Hallucination detected")
```

### 🎯 Interview Line

> “We use an LLM as a judge to verify grounding before responding.”

---

# 🏗️ End-to-End Hallucination Guard (Production Pattern)

```python
def safe_rag_response(query):
    docs = fusion_retriever.get_relevant_documents(query)
    answer = generate_answer(query)[0]

    if semantic_faithfulness_check(answer, docs) < 0.75:
        return "I cannot verify this information."

    if not llm_faithfulness_check(answer, docs):
        return "The answer could not be validated."

    return answer
```

---

# ❌ What Happens When Hallucination Is Detected?

✔ Reject the answer
✔ Ask clarifying question
✔ Re-run retrieval (increase k / use RAG Fusion)
✔ Return partial or safe response

---

# 🔥 Interview-Winning Summary (Say This)

> “We detect hallucinations using a layered approach: fast keyword grounding, semantic similarity checks, and LLM-based faithfulness evaluation. If validation fails, we reject or regenerate the answer. This ensures responses are always grounded in retrieved evidence.”

---

# 🧠 What Interviewers LOVE

✔ You don’t trust LLM blindly
✔ You reject answers
✔ You mention semantic similarity
✔ You use LLM as judge
✔ You focus on retrieval quality

---

## 🚀 Want Next?

I can give you:

* ✅ **Hallucination detection architecture diagram**
* ✅ **Metrics used in real projects**
* ✅ **Bad vs good answers comparison**
* ✅ **System design interview answer**

Just say **NEXT**
